# UE Data Mining - Projet

Notebook à compléter dans le cadre du projet de l’UE Data Mining des Master 2 Intelligence Artificielle et Master 2 Data Science de l’année scolaire 2026-2027.

* Nom du groupe : 12
* Membres du groupe :
    * idil MAHAMOUD GUEDI
    * André da Conceição
    * Ali Chabaka 
    * Aissam Eddine Boukhelkhal
    * Lucas Duchamp
    * RATOANDROMANANA Iavosoa Miella


## Pensez à mettre des titres

Et du texte pour expliquer les différentes étapes de votre raisonnement, les résultats obtenus et les conclusions que vous en avez tiré.

Pour rappel, ce notebook sera aussi votre rapport sur ce projet, donc soyez précis et rigoureux dans ce que vous décrivez et écrivez.

In [29]:
### Data Cleaning
### Ecology
import pandas as pd
import re
df_bats = pd.read_csv('data/ecology/bat_surveys.csv')
df_env_field = pd.read_csv('data/ecology/environmental_field_reports.csv')
df_env_monthly = pd.read_csv('data/ecology/environmental_monthly.csv')
df_monitor_dev = pd.read_csv('data/ecology/monitoring_device_operations.csv')
df_monitor_sites = pd.read_csv('data/ecology/monitoring_sites.csv')
df_orchard = pd.read_csv('data/ecology/orchard_surveys.csv')

In [ ]:
### bats_survey
df_bats.describe()

In [ ]:
# On enlève les lignes doublons (il y en a 2)
print(df_bats.duplicated().sum())
df_bats_clean = df_bats.drop_duplicates()

In [ ]:
# On remplace les noms des observer team comme "unknown")
print(df_bats_clean.isna().sum())
df_bats_clean['observer_team'] = df_bats_clean['observer_team'].fillna('unknown')

In [ ]:
for c in df_bats_clean:
    print(df_bats_clean[c].value_counts())

# On observe problèmes de generalisation de dates, de ponctuations (floats), de majuscules etc
# On a aussi la survey_quality à mettre en 2 ou 3 colonnes

In [ ]:
def clean_numeric(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    s = re.sub(r'[^\d,.\-]', '', s)
    # on enleve les lettres
    s = s.replace(',', '.')
    # on remplace les virgules
    try:
        return float(s)
    except ValueError:
        print(s)
        return None
for c in ["observation_hours","observed_bat_count","acoustic_detection_count","nightly_visit_count","colony_estimate","comparison_species_count","weather_interference"]:
    df_bats_clean[c] = df_bats_clean[c].apply(clean_numeric)
print(df_bats_clean)

In [ ]:
df_bats_clean["survey_method"] = df_bats_clean['survey_method'].str.lower().str.strip()

df_bats_clean["survey_end"] = pd.to_datetime(df_bats_clean['survey_end'],errors="coerce")
df_bats_clean['survey_end'] = df_bats_clean['survey_end'].dt.strftime('%Y-%m-%dT%H:%M:%S')

print(df_bats_clean["survey_method"].value_counts())
print(df_bats_clean["survey_end"].value_counts())

# Attention : on a 63 valeurs au lieu de 64 comme tout le reste

In [ ]:
# On créé 3 catégories à partir de survey_quality
# On garde la colonne survey_quality
dummies = pd.get_dummies(df_bats_clean['survey_quality'], prefix='survey_quality')
df_bats_clean = pd.concat([df_bats_clean, dummies], axis=1)

In [ ]:
for c in df_bats_clean:
    print(df_bats_clean[c].value_counts())

In [ ]:
### env_field
# On remplace un nom manquant dans author_name comme "unknown")
print(df_env_field.isna().sum())
df_env_field['author_name'] = df_env_field['author_name'].fillna('unknown')

In [ ]:
### env_monthly
df_env_monthly.describe()

In [ ]:
# On enlève les lignes doublons (il y en a 2 les mêmes que bats_survey)
print(df_env_monthly.duplicated())
df_env_monthly = df_env_monthly.drop_duplicates()

In [ ]:
# On remarque dans describe que 3 colonnes n'ont pas été traitées donc pas uniformisées
for c in ["mean_temperature_c","rainfall_mm","mean_humidity_pct"]:
    df_env_monthly[c] = df_env_monthly[c].apply(clean_numeric)

df_env_monthly.describe()

In [ ]:
# On doit décider comment gérer les valeurs manquantes
df_env_monthly.dropna(subset=["vegetation_cover_pct","pesticide_index"] ,inplace=True)
print( df_env_monthly.isna().sum())

In [ ]:
# On créé 2 catégories à partir de weather_exception_code
# On garde la colonne weather_exception_code
df_env_monthly["weather_exception_code"].value_counts()
dummies = pd.get_dummies(df_env_monthly['weather_exception_code'],
                         prefix='weather_exception_code')
df_env_monthly = pd.concat([df_env_monthly, dummies], axis=1)

In [ ]:
df_env_monthly.describe()

In [ ]:
### Monitoring device operations
df_monitor_dev["uptime_pct"] = df_monitor_dev["uptime_pct"].apply(clean_numeric)
df_monitor_dev = df_monitor_dev.drop_duplicates()

In [ ]:
df_monitor_dev.describe()

In [ ]:
# On créé 2 catégories à partir de fault_code
# On garde la colonne fault_code
df_monitor_dev["fault_code"].value_counts()
dummies = pd.get_dummies(df_monitor_dev['fault_code'],
                         prefix='fault_code')
df_monitor_dev = pd.concat([df_monitor_dev, dummies], axis=1)

In [30]:
### Monitoring sites
df_monitor_sites = df_monitor_sites.drop_duplicates()
df_monitor_sites = df_monitor_sites.drop(columns="programme_end_date")


In [31]:
# On créé 2 catégories à partir de cave_access_category
df_monitor_sites["cave_access_category"].value_counts()
dummies = pd.get_dummies(df_monitor_sites['cave_access_category'],
                         prefix='cave_access_category')
df_monitor_sites = pd.concat([df_monitor_sites, dummies], axis=1)
# On créé 2 catégories à partir de orchard_context
df_monitor_sites["orchard_context"].value_counts()
dummies = pd.get_dummies(df_monitor_sites['orchard_context'],
                         prefix='orchard_context')
df_monitor_sites = pd.concat([df_monitor_sites, dummies], axis=1)
# On pourrait le faire aussi pour habitat category

In [35]:
for c in df_monitor_sites:
    print(df_monitor_sites[c].value_counts())

site_code
KNT-KES    1
KNT-LUM    1
KNT-SEN    1
AVL-MAR    1
AVL-VEL    1
          ..
TMB-WST    1
TMB-SXP    1
TMB-WQL    1
TMB-QCB    1
TMB-DLQ    1
Name: count, Length: 64, dtype: int64
site_name
Kestrel Cavern Monitoring Site    1
Luma Karst Reserve Site           1
Senna Ridge Cave Site             1
Marula East Orchard Site          1
Velin Basin Orchard Site          1
                                 ..
Garnet Valley Monitoring Site     1
Juniper Grove Monitoring Site     1
Mango Coast Monitoring Site       1
Pandan Ridge Monitoring Site      1
Sable Forest Monitoring Site      1
Name: count, Length: 64, dtype: int64
site_category
habitat monitoring    40
orchard monitoring    15
cave monitoring        8
ORCHARD MONITORING     1
Name: count, dtype: int64
region
Kintara Karst Cluster                   22
Eastern Avelora Agricultural Cluster    22
Tembara Coastal Cluster                 20
Name: count, dtype: int64
jurisdiction
Kintara Republic        22
Avelora Federation     